### Wave D — Time to convert

Sample: Month × `0-7 / 8-15 / 16-30 / 31-60 / 60+` days (born → Sales Closure).

| Sheet | Yellow box |
|-------|------------|
| `Time_To_Convert_Buckets` | #18–19 sample |
| `Duration_By_Stage` | stage timing |



In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path("/Users/lekhanigam/data/leads_sv_closure_reports")
sys.path.insert(0, str(ROOT))
from common import START_TS, END_TS, CLOSURE_STAGE, q, write_excel

born = q(f"""
SELECT CAST(_id AS varchar) lead_oid, createdat born_at,
       date_format(createdat, '%Y-%m') month_key, date_format(createdat, '%M') month_name
FROM leads WHERE createdat >= {START_TS} AND createdat < {END_TS}
""")
closed = q(f"""
SELECT CAST(lead AS varchar) lead_oid, MIN(createdat) closed_at
FROM leadstagehistories
WHERE CAST(stage AS varchar)='{CLOSURE_STAGE}' AND createdat >= {START_TS} AND createdat < {END_TS}
GROUP BY 1
""")
m = born.merge(closed, on="lead_oid", how="inner")
m["days"] = (pd.to_datetime(m["closed_at"]) - pd.to_datetime(m["born_at"])).dt.total_seconds() / 86400.0

def bucket(d):
    if d <= 7: return "0-7 Days"
    if d <= 15: return "8-15 Days"
    if d <= 30: return "16-30 Days"
    if d <= 60: return "31-60 Days"
    return "60+ Days"

m["bucket"] = m["days"].map(bucket)
order = ["0-7 Days", "8-15 Days", "16-30 Days", "31-60 Days", "60+ Days"]
buckets = (
    m.groupby(["month_key", "month_name", "bucket"], as_index=False).agg(n=("lead_oid", "nunique"))
    .pivot_table(index=["month_key", "month_name"], columns="bucket", values="n", aggfunc="sum", fill_value=0)
    .reset_index().sort_values("month_key")
)
for b in order:
    if b not in buckets.columns: buckets[b] = 0
buckets = buckets.rename(columns={"month_name": "Month"})[["Month"] + order]

stages = q(f"""
SELECT CAST(lead AS varchar) lead_oid, CAST(stage AS varchar) stage, MIN(createdat) first_at
FROM leadstagehistories
WHERE createdat >= {START_TS} AND createdat < {END_TS}
  AND CAST(stage AS varchar) IN ('Lead Initiated','Pre Site Visit','Post Site Visit','Sales Closure','Not Interested')
GROUP BY 1, 2
""")
st = stages.merge(born[["lead_oid", "born_at"]], on="lead_oid", how="inner")
st["days"] = (pd.to_datetime(st["first_at"]) - pd.to_datetime(st["born_at"])).dt.total_seconds() / 86400.0
dur = st.groupby("stage", as_index=False).agg(n=("lead_oid", "nunique"), median_days=("days", "median"), mean_days=("days", "mean"))
dur["median_days"] = dur["median_days"].round(1)
dur["mean_days"] = dur["mean_days"].round(1)

write_excel(ROOT / "Lead_Time_To_Convert_Analysis.xlsx", {
    "Time_To_Convert_Buckets": buckets,
    "Duration_By_Stage": dur.sort_values("median_days"),
})
print("closures timed", len(m))

